# Task 3: Classical Language Modeling (20%)

## Task 3.1 – N-gram Language Models
For this task you should implement a bigram and a trigram language model
Requirements:

- Compute smoothed probabilities (e.g., Laplace smoothing)
- Generate short text sequences from each model
- Compute and compare perplexity across n-gram sizes

In [5]:
import math
import re
import random
from collections import Counter
import pandas as pd


class NgramLanguageModel:
    """
    Simple n-gram LM with Laplace (add-alpha) smoothing.

    P(w | context) = (count(context,w) + alpha) / (count(context) + alpha*V)
    """

    def __init__(self, n: int, alpha: float = 1.0, unk_token: str = "<unk>"):
        assert n >= 1
        self.n = n
        self.alpha = alpha
        self.unk = unk_token

        self.vocab = set()
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.V = 0
        self.fitted = False

    @staticmethod
    def tokenize(text: str) -> list[str]:
        """
        A simple tokenizer:
        - lowercase
        - keeps punctuation as separate tokens
        """
        text = (text or "").lower()
        # words with optional apostrophes, numbers, or single punctuation symbols
        return re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+|[^\w\s]", text)

    def _prepare_tokens(self, tokens: list[str]) -> list[str]:
        pads = ["<s>"] * (self.n - 1)
        return pads + tokens + ["</s>"]

    def fit(self, texts: list[str]) -> None:
        # 1) build vocabulary from training texts
        for t in texts:
            self.vocab.update(self.tokenize(t))

        # add special tokens
        self.vocab.update({"<s>", "</s>", self.unk})
        self.V = len(self.vocab)

        # 2) count ngrams + contexts
        for t in texts:
            tokens = self.tokenize(t)
            tokens = [tok if tok in self.vocab else self.unk for tok in tokens]
            sent = self._prepare_tokens(tokens)

            for i in range(self.n - 1, len(sent)):
                ngram = tuple(sent[i - self.n + 1 : i + 1])
                ctx = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[ctx] += 1

        self.fitted = True

    def prob(self, word: str, context: list[str]) -> float:
        if not self.fitted:
            raise RuntimeError("Model not fitted. Call fit() first.")

        if word not in self.vocab:
            word = self.unk

        if len(context) != self.n - 1:
            raise ValueError(f"context length must be {self.n - 1} for {self.n}-gram model")

        ctx = tuple(context)
        num = self.ngram_counts[ctx + (word,)] + self.alpha
        den = self.context_counts[ctx] + self.alpha * self.V
        return num / den

    def generate(self, max_tokens: int = 30, seed: int | None = None) -> str:
        """
        Generate a short sequence by sampling from P(next | context).
        """
        if not self.fitted:
            raise RuntimeError("Model not fitted. Call fit() first.")
        if seed is not None:
            random.seed(seed)

        context = ["<s>"] * (self.n - 1)
        out = []

        vocab_list = list(self.vocab)

        for _ in range(max_tokens):
            ctx = tuple(context)
            den = self.context_counts[ctx] + self.alpha * self.V

            weights = [
                (self.ngram_counts[ctx + (w,)] + self.alpha) / den
                for w in vocab_list
            ]
            w = random.choices(vocab_list, weights=weights, k=1)[0]

            if w == "</s>":
                break

            out.append(w)

            if self.n > 1:
                context = (context + [w])[-(self.n - 1):]

        return " ".join(out)

    def perplexity(self, texts: list[str]) -> float:
        """
        Perplexity = exp( - (1/N) * sum log P(w_i | context_i) )
        """
        if not self.fitted:
            raise RuntimeError("Model not fitted. Call fit() first.")

        logp_sum = 0.0
        N = 0

        for t in texts:
            tokens = self.tokenize(t)
            tokens = [tok if tok in self.vocab else self.unk for tok in tokens]
            sent = self._prepare_tokens(tokens)

            for i in range(self.n - 1, len(sent)):
                ctx = sent[i - self.n + 1 : i]
                w = sent[i]
                p = self.prob(w, ctx)
                logp_sum += math.log(p)
                N += 1

        return math.exp(-logp_sum / N) if N > 0 else float("inf")


In [7]:
class Task3NgramLanguageModel:
    """
    Wrapper for Task 3.1:
    - load train/test csv
    - train bigram + trigram with Laplace smoothing
    - generate samples
    - compute perplexities
    """

    def __init__(self, alpha: float = 1.0, use_columns=("Title", "Description")):
        self.alpha = alpha
        self.use_columns = use_columns
        self.bigram = NgramLanguageModel(n=2, alpha=alpha)
        self.trigram = NgramLanguageModel(n=3, alpha=alpha)

    def _load_texts(self, csv_path: str) -> list[str]:
        df = pd.read_csv(csv_path)
        for c in self.use_columns:
            if c not in df.columns:
                raise ValueError(f"Missing column '{c}' in {csv_path}. Found: {list(df.columns)}")
        text = df[self.use_columns[0]].fillna("").astype(str)
        for c in self.use_columns[1:]:
            text = text + " " + df[c].fillna("").astype(str)
        return text.tolist()

    def train(self, train_csv: str) -> None:
        train_texts = self._load_texts(train_csv)
        self.bigram.fit(train_texts)
        self.trigram.fit(train_texts)

    def evaluate(self, test_csv: str) -> dict:
        test_texts = self._load_texts(test_csv)
        pp2 = self.bigram.perplexity(test_texts)
        pp3 = self.trigram.perplexity(test_texts)
        return {"bigram_perplexity": pp2, "trigram_perplexity": pp3}

    def demo_generation(self, k: int = 3, max_tokens: int = 30, seed: int = 0) -> dict:
        bigram_samples = [self.bigram.generate(max_tokens=max_tokens, seed=seed + i) for i in range(k)]
        trigram_samples = [self.trigram.generate(max_tokens=max_tokens, seed=seed + i) for i in range(k)]
        return {"bigram": bigram_samples, "trigram": trigram_samples}


In [8]:

task = Task3NgramLanguageModel(alpha=1.0)  # Laplace smoothing (alpha=1)
task.train("train.csv")

print(task.demo_generation(k=3, max_tokens=25, seed=42))

results = task.evaluate("test.csv")
print(results)


{'bigram': ['stunning wedlock frode cobol soliz spellbinding domecq possessing signing sandis lax mkm rivermen fiber headlong clintons cheesy shellshock joys sunderland pettersson insecurity burdines gannett talkingpointsmemo', "parly clich latter's bushmeat sweatshirts toasted idling kpu heap nabisco outmaneuvering excavate alacrity kyoto ammonia centrist quicktest sols tnaiste skateboarder cian upping mobbed made clarkston", "qtype 6620 foundry strives riley's resentencing eurosport onxx filice lenka interval castel wife's mie hampered trek hange constitutional donzi itm societe rolle bedford livid shareholder"], 'trigram': ["stunning want frode ringside identify spellbinding domecq china's composure elicit lax mkm rivermen fiber goosman novell cheesy 010 joys sunderland pettersson 647 romeoville widens trials", "parly toenail latter's nepalis ethiopia toasted fptozdumb kpu heap nabisco outmaneuvering excavate acid personalied xvi propellant peerage movers banda asensio cian willams 

## Task 3.2 – Comparison with a Neural Language Model
Build a simple neural language model consisting of:

- An embedding layer
- One LSTM or GRU layer
- A softmax output layer

Then compare against n-gram models on:

- Perplexity
- Quality of generated text
- And, Training time

In [3]:
import math
import re
import time
from dataclasses import dataclass
from typing import List, Tuple

import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


# -----------------------------
# 1) Tokenizer（和你 n-gram 统一）
# -----------------------------
def tokenize(text: str) -> List[str]:
    text = (text or "").lower()
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+|[^\w\s]", text)


# -----------------------------
# 2) 读 CSV -> list[str]
# -----------------------------
def load_texts(csv_path: str, use_columns=("Title", "Description")) -> List[str]:
    df = pd.read_csv(csv_path)
    for c in use_columns:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {csv_path}. Found: {list(df.columns)}")

    text = df[use_columns[0]].fillna("").astype(str)
    for c in use_columns[1:]:
        text = text + " " + df[c].fillna("").astype(str)
    return text.tolist()


# -----------------------------
# 3) 词表
# -----------------------------
@dataclass
class Vocab:
    stoi: dict
    itos: list
    pad: str = "<pad>"
    bos: str = "<s>"
    eos: str = "</s>"
    unk: str = "<unk>"

    @property
    def pad_id(self): return self.stoi[self.pad]
    @property
    def bos_id(self): return self.stoi[self.bos]
    @property
    def eos_id(self): return self.stoi[self.eos]
    @property
    def unk_id(self): return self.stoi[self.unk]
    @property
    def size(self): return len(self.itos)

    def encode_tokens(self, toks: List[str]) -> List[int]:
        return [self.stoi.get(t, self.unk_id) for t in toks]

    def decode_ids(self, ids: List[int]) -> List[str]:
        return [self.itos[i] for i in ids]


def build_vocab(texts: List[str], min_freq: int = 1) -> Vocab:
    from collections import Counter
    cnt = Counter()
    for t in texts:
        cnt.update(tokenize(t))

    specials = ["<pad>", "<s>", "</s>", "<unk>"]
    itos = specials[:]
    for w, f in cnt.items():
        if f >= min_freq and w not in specials:
            itos.append(w)

    stoi = {w: i for i, w in enumerate(itos)}
    return Vocab(stoi=stoi, itos=itos)


# -----------------------------
# 4) 把整段文本变成“连续 token 流”
#    然后做 next-token prediction
# -----------------------------
def texts_to_token_ids(texts: List[str], vocab: Vocab) -> List[int]:
    ids = []
    for t in texts:
        toks = [vocab.bos] + tokenize(t) + [vocab.eos]
        ids.extend(vocab.encode_tokens(toks))
    return ids


class LMSequenceDataset(Dataset):
    """
    给定一个 token id 序列 stream：
    取长度 seq_len 的输入 x
    预测后面一个 token 的序列 y（右移一位）
    """
    def __init__(self, token_ids: List[int], seq_len: int = 32):
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        # 每个样本需要 seq_len+1 个 token（因为要做 next token）
        return max(0, len(self.data) - (self.seq_len + 1))

    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.seq_len + 1]
        x = chunk[:-1]   # 输入
        y = chunk[1:]    # 目标（右移一位）
        return x, y


# -----------------------------
# 5) 神经语言模型：Embedding + (GRU/LSTM) + Linear
# -----------------------------
class NeuralLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, hidden_dim: int = 256,
                 rnn_type: str = "gru", num_layers: int = 1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        if rnn_type.lower() == "lstm":
            self.rnn = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: [B, T]
        e = self.emb(x)            # [B, T, E]
        h, _ = self.rnn(e)         # [B, T, H]
        logits = self.fc(h)        # [B, T, V]
        return logits


# -----------------------------
# 6) 训练 / 评估 perplexity / 生成
# -----------------------------
def train_neural_lm(
    train_ids: List[int],
    vocab: Vocab,
    seq_len: int = 32,
    batch_size: int = 64,
    epochs: int = 2,
    lr: float = 2e-3,
    rnn_type: str = "gru",
    device: str = "cpu",
):
    ds = LMSequenceDataset(train_ids, seq_len=seq_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)

    model = NeuralLM(vocab_size=vocab.size, rnn_type=rnn_type).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()  # Includes softmax + NLL (for more stable values)

    t0 = time.perf_counter()
    model.train()
    for ep in range(epochs):
        total_loss = 0.0
        steps = 0
        for x, y in dl:
            x = x.to(device)  # [B, T]
            y = y.to(device)  # [B, T]
            logits = model(x) # [B, T, V]

            # The CrossEntropyLoss input requires [N, C] and [N] respectively.
            loss = crit(logits.reshape(-1, vocab.size), y.reshape(-1))

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()
            steps += 1

        avg_loss = total_loss / max(1, steps)
        ppl = math.exp(avg_loss)
        print(f"Epoch {ep+1}/{epochs} - loss={avg_loss:.4f} - ppl={ppl:.2f}")

    train_time = time.perf_counter() - t0
    return model, train_time


@torch.no_grad()
def perplexity_neural_lm(model: nn.Module, test_ids: List[int], vocab: Vocab,
                         seq_len: int = 32, batch_size: int = 64, device: str = "cpu") -> float:
    ds = LMSequenceDataset(test_ids, seq_len=seq_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)
    
    # Accumulated total loss makes it easier to calculate the average.
    crit = nn.CrossEntropyLoss(reduction="sum") 
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for x, y in dl:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = crit(logits.reshape(-1, vocab.size), y.reshape(-1))
        total_loss += loss.item()
        total_tokens += y.numel()

    avg_nll = total_loss / max(1, total_tokens)
    return math.exp(avg_nll)


@torch.no_grad()
def generate_neural_lm(model: nn.Module, vocab: Vocab, max_tokens: int = 30,
                       temperature: float = 1.0, device: str = "cpu", seed: int = 0) -> str:
    torch.manual_seed(seed)

    # Start from <s>
    context = torch.tensor([[vocab.bos_id]], dtype=torch.long, device=device)

    out_tokens = []
    model.eval()

    for _ in range(max_tokens):
        logits = model(context)              # [1, T, V]
        next_logits = logits[:, -1, :]       # [1, V]
        next_logits = next_logits / max(1e-6, temperature)

        probs = torch.softmax(next_logits, dim=-1)  # [1, V]
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == vocab.eos_id:
            break

        out_tokens.append(vocab.itos[next_id])

        # Append the generated token to the context (continuously increasing).
        context = torch.cat([context, torch.tensor([[next_id]], device=device)], dim=1)

    return " ".join(out_tokens)


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_texts = load_texts("train.csv")[:10000]
test_texts  = load_texts("test.csv")[:10000]

vocab = build_vocab(train_texts, min_freq=1)

train_ids = texts_to_token_ids(train_texts, vocab)
test_ids  = texts_to_token_ids(test_texts, vocab)

model, train_time = train_neural_lm(
    train_ids, vocab,
    seq_len=32, batch_size=64,
    epochs=2, lr=2e-3,
    rnn_type="gru",  # or "lstm"
    device=device
)

pp_nn = perplexity_neural_lm(model, test_ids, vocab, seq_len=32, batch_size=64, device=device)
print("Neural LM perplexity:", pp_nn)
print("Neural LM training time (s):", train_time)

print("Sample generation 1:", generate_neural_lm(model, vocab, max_tokens=25, temperature=1.0, device=device, seed=42))
print("Sample generation 2:", generate_neural_lm(model, vocab, max_tokens=25, temperature=0.8, device=device, seed=43))

Epoch 1/2 - loss=2.4482 - ppl=11.57
Epoch 2/2 - loss=1.2007 - ppl=3.32
Neural LM perplexity: 17790.870518244108
Neural LM training time (s): 4306.7814671250235
Sample generation 1: china change in order to china beijing ( reuters ) - oil exports by the world's largest economy .
Sample generation 2: police defuse bomb on beagle shrine najaf , iraq ( reuters ) - a radical shi # 39 ; ite cleric said friday .
